# M7 · Lab 4 — Explainability with SHAP (and where LIME fits)
### Module 7 — Feature Stores, Experiment Management & Explainability | Spine: Truck Delay Classification

| | |
|---|---|
| **Duration** | 75 min (incl. ~15 min concept pre-read) · **Difficulty** Intermediate · **Tier 3** |
| **Tools** | Python 3.12.10, `shap`, `xgboost`, `matplotlib` — the **real** M3 model |
| **Prerequisite** | None beyond the shipped artifacts in `labs/data/` |
| **You answer** | "**Why** did the model predict *delayed* for *this* shipment?" — what regulators + ops ask for |

## 💻 Where to run this
Run on the **same SageMaker notebook instance from M6** (auto-stops overnight — just press **Start**). New deps (`shap`,
`matplotlib`) install in the setup cell. **Portable fallback:** Colab / local Jupyter on **Python 3.12.10**. The **real**
M3 artifacts ship in `data/` — nothing synthetic. (No AWS cost — SHAP runs locally.)

## 🎯 What you'll be able to do after this lab
1. **Explain** what model explainability/interpretability *is* and **why** it's non-negotiable in production.
2. **Map the landscape** of explainability techniques — intrinsic vs post-hoc, global vs local, model-specific vs
   model-agnostic — and place **LIME** and **SHAP** within it.
3. **Describe how SHAP values are computed** (Shapley values from game theory) and why `TreeExplainer` is exact + fast for XGBoost.
4. **Read every SHAP output** — base value, additivity, beeswarm, bar, **waterfall/force**, dependence.
5. **Explain two concrete predictions** end-to-end (one *delayed*, one *on-time*) — turning SHAP numbers into a sentence an
   ops lead or auditor accepts — and **connect SHAP importance back to M6 drift severity**.

Concept sections 0a–0f are a one-time **pre-read**; the rest is hands-on with numeric, inspectable outputs.

---
## 0a · What is explainability — and why do we need it?

A trained XGBoost with 128 features and hundreds of trees is a **black box**: it outputs `delayed = 0.77` but says nothing
about *why*. **Explainability** (a.k.a. interpretability) is the set of techniques that answer **"which inputs drove this
output, and by how much?"** — at two levels:
- **Global:** what does the model rely on *overall*? (Across all shipments, does weather matter more than truck age?)
- **Local:** why did the model predict *this one* shipment the way it did?

Why this is mandatory, not a nice-to-have:

| Driver | What it needs from explainability |
|---|---|
| **Trust & adoption** | An ops lead won't act on "the model said so." They act on "flagged because precip is high *and* the route is long." |
| **Debugging** | If the model leans on a leaked or silly feature, only an explanation reveals it. |
| **Fairness / bias** | Show the decision isn't driven by a protected attribute (gender, etc.). |
| **Regulation / compliance** | Banking & GDPR's "right to an explanation" require a **per-decision** reason. This is the FreshBasket finance context — auditable explanations are required. |
| **Actionability** | "Reduce the delay risk by re-routing around the storm cell" only exists if you know precip drove the flag. |

> **The one-line stakes:** a model you can't explain is a model you can't deploy in a regulated or high-stakes setting —
> no matter how accurate it is.

## 0b · The landscape of explainability techniques

Explainability methods sort along three axes:
- **Intrinsic vs Post-hoc** — *intrinsic* = the model is transparent by design (linear/logistic regression coefficients,
  shallow decision trees, GAMs). *Post-hoc* = explain an already-trained black box from the outside. (XGBoost → post-hoc.)
- **Global vs Local** — the whole model vs a single prediction.
- **Model-specific vs Model-agnostic** — tied to one model family vs works on *any* model (treats it as `f(x)`).

The common toolkit:

| Technique | Scope | Agnostic? | One-liner | Watch-out |
|---|---|---|---|---|
| **Model coefficients** | Global+Local | No (linear) | Read the weights directly | Only for linear/GLM models |
| **Tree `feature_importances_`** | Global | No (trees) | Built-in gain/split counts | **Inconsistent & biased** toward high-cardinality features |
| **Permutation importance** | Global | ✅ | Shuffle a feature, measure score drop | Misleads under correlated features |
| **PDP / ICE plots** | Global (effect) | ✅ | Average effect of a feature across its range | Assumes feature independence |
| **LIME** | **Local** | ✅ | Fit a simple model *around* one point | Unstable; local only |
| **SHAP** | **Local + Global** | ✅ | Game-theoretic fair attribution | Slower (but exact + fast for trees) |

The two **post-hoc, model-agnostic, local** heavyweights — the ones you'll be asked about — are **LIME** and **SHAP**.

## 0c · LIME — Local Interpretable Model-agnostic Explanations

**Idea:** to explain one prediction, **LIME** doesn't try to understand the whole model. It **perturbs** the instance
(jitters its feature values to create many nearby fake points), asks the black box for predictions on them, and then fits a
**simple, interpretable surrogate** (usually a weighted linear regression) that mimics the black box *in that small
neighbourhood*. The surrogate's coefficients are the explanation: "locally, increasing precip pushes this prediction up."

| 👍 Strengths | 👎 Weaknesses |
|---|---|
| Fast, intuitive, truly model-agnostic | **Unstable** — re-run it and the explanation can change (random sampling) |
| Works on tabular, text, images | **Local only** — no consistent global view |
| Easy to grasp (it's just a local linear fit) | The "neighbourhood" definition is a hyperparameter that changes results |

In Python it's the `lime` package (`pip install lime`). We *describe* LIME here for contrast but **use SHAP** for the
hands-on, because SHAP gives the same local story **plus** a consistent global one — and it's grounded in theory.

## 0d · SHAP — SHapley Additive exPlanations

**SHAP** explains a prediction by borrowing a result from cooperative **game theory** (Lloyd Shapley, 1953). Set up the game:
- The **players** = the features.
- The **payout** = the model's prediction for this row, measured *relative to the average prediction* (the **base value**).
- Each feature's **SHAP value** = its **fair share** of that payout — its average marginal contribution across *every*
  possible order in which features could be added to the model.

Why "fair"? Shapley values are the **unique** attribution satisfying these axioms, which is what makes SHAP trustworthy:

| Property | Plain meaning |
|---|---|
| **Local accuracy / additivity** | `base value + Σ(SHAP values) = the model's output for this row` — the explanation *reconstructs* the prediction exactly |
| **Missingness** | A feature that didn't change the output gets 0 credit |
| **Consistency** | If a model changes so a feature matters more, its SHAP value can't go down — the property tree `feature_importances_` violates |

**SHAP vs LIME in one line:** LIME fits a local approximation and *hopes* it's faithful; SHAP computes the *provably fair*
contribution and is additive — so SHAP gives you a reliable **local *and* global** picture. That additivity (Step 3) is the
property that lets SHAP literally **explain the predicted label** as a sum of feature pushes.

## 0e · How are SHAP values actually calculated?

**The exact definition.** For a feature *i*, its Shapley value averages the *marginal contribution* of adding *i* over all
subsets (coalitions) *S* of the other features:

> φᵢ = Σ over coalitions S   [ |S|! · (n−|S|−1)! / n! ] · ( f(S ∪ {i}) − f(S) )

In words: "across every way the other features could already be 'in the model', how much does adding feature *i* change the
prediction, on average?" The weights make every *ordering* of features count equally.

**A tiny worked intuition (2 features: Rain R, Traffic T).** Let the prediction rise by **a** when only R is known, by **b**
when only T is known, and by **c** when both are known (c may ≠ a+b — that gap is *interaction*). Averaging over the two
orders:
- φ_R = ½·a + ½·(c − b)   ·   φ_T = ½·b + ½·(c − a)
- and **φ_R + φ_T = c** — the contributions add up to the total change from the base. That's additivity in miniature.

**The catch:** exact Shapley needs all 2ⁿ coalitions — impossible for 128 features. So SHAP ships **specialised explainers**:

| Explainer | For | Cost |
|---|---|---|
| **`TreeExplainer`** | tree models (XGBoost, LightGBM, RF) | **Exact** + fast — *polynomial*, not 2ⁿ (we use this) |
| `LinearExplainer` | linear models | Exact, trivial |
| `DeepExplainer` / `GradientExplainer` | neural nets | Approximate, gradient-based |
| `KernelExplainer` | **any** model (model-agnostic) | Approximate, **slow** (LIME-like sampling) |

Because our model is XGBoost, `TreeExplainer` gives us **exact** Shapley values quickly — the best of both worlds.

## 0f · How to read the SHAP outputs (your cheat-sheet for the plots below)

SHAP values here are in **log-odds (margin)** units — positive pushes toward **delayed**, negative toward **on-time**.

- **Base value** `E[f(x)]` — the model's average output over the data (≈ the overall log-odds of delay). Every explanation
  starts here and the feature pushes move it to this row's prediction.
- **Beeswarm (summary) plot** — *global*. One dot per row per feature; **x = SHAP value** (impact), **colour = feature
  value** (red high / blue low). Features sorted by overall importance. Read: "high precip (red dots) sits on the positive
  side → high precip increases delay risk."
- **Bar plot** — *global*. Mean |SHAP| per feature = a clean importance ranking.
- **Waterfall / Force plot** — *local*. Starts at the base value and stacks each feature's push (red = toward delayed, blue
  = toward on-time) until it reaches `f(x)` for this row. **This is the chart you show an ops lead or auditor.**
- **Dependence plot** — *global-ish*. SHAP value of one feature vs its actual value → the shape of its effect, coloured by
  an interacting feature.

---
## Step 1 · Setup + build the real model matrix

**What we're doing:** installing `shap`, loading the **real M3 model + data**, and rebuilding the exact **128-feature**
matrix the model expects (M3's scaler + one-hot encoder). **Why a sample of 1,000 rows:** SHAP explains per-row; 1,000 real
rows give a stable *global* picture fast, and we'll drill into individual rows for the *local* story.

In [1]:
import sys, subprocess
def _pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
try:
    import shap
except ImportError:
    _pip("shap", "matplotlib"); import shap
import matplotlib.pyplot as plt
print("shap", shap.__version__)   # modern shap (>=0.44) supports numpy 2; no numpy<2 pin needed

shap 0.46.0

In [2]:
import os, zipfile
DATA_DIR = os.environ.get("M7_DATA_DIR", "data")
if not os.path.isdir(DATA_DIR) and os.path.exists("data.zip"):
    with zipfile.ZipFile("data.zip") as z: z.extractall(".")
    print("Extracted data.zip ->", DATA_DIR + "/")
else:
    print("Data already present in", DATA_DIR + "/")

Data already present in data/

In [3]:
import json
import numpy as np, pandas as pd

REF_CSV = os.path.join(DATA_DIR, "reference", "final_features.csv")
ART_DIR = os.path.join(DATA_DIR, "artifacts")
ref   = pd.read_csv(REF_CSV)
fmeta = json.load(open(os.path.join(DATA_DIR, "reference", "feature_metadata.json")))
TARGET = fmeta["target"]
print("Reference frame:", ref.shape, "| delay rate:", round(ref[TARGET].mean(), 4))

Reference frame: (12308, 37) | delay rate: 0.3489

In [4]:
import joblib
_model   = joblib.load(os.path.join(ART_DIR, "xgboost_model.pkl"))
_encoder = joblib.load(os.path.join(ART_DIR, "encoder.pkl"))
_scaler  = joblib.load(os.path.join(ART_DIR, "scaler.pkl"))
mmeta = json.load(open(os.path.join(ART_DIR, "model_metadata.json")))
M_CONT, M_CAT, M_BIN, M_FEATS = (mmeta["continuous_cols"], mmeta["categorical_cols"],
                                 mmeta["binary_ordinal_cols"], mmeta["feature_names"])

def to_model_matrix(df):
    x_cont = pd.DataFrame(_scaler.transform(df[M_CONT]), columns=M_CONT)
    x_cat  = pd.DataFrame(_encoder.transform(df[M_CAT]), columns=_encoder.get_feature_names_out(M_CAT))
    x_bin  = df[M_BIN].reset_index(drop=True)
    return pd.concat([x_cont, x_cat, x_bin], axis=1)[M_FEATS]
print("Preprocessing helper ready -> produces the", len(M_FEATS), "feature model matrix.")

Preprocessing helper ready -> produces the 128 feature model matrix.

In [5]:
# Explain on a representative sample of the REAL data; keep the RAW rows too (for human-readable values later).
sample = ref.sample(1000, random_state=42).reset_index(drop=True)
X = to_model_matrix(sample)
print("Explaining", X.shape[0], "real rows x", X.shape[1], "features")

Explaining 1000 real rows x 128 features

**✅ Result:** we have a **1,000 × 128** matrix of real shipments ready to explain, plus the matching **raw** rows in
`sample` (so we can say "precip = 0.8 inches", not the scaled value the model sees). On to the explanations.

---
## Step 2 · Compute SHAP values with `TreeExplainer`

**What we're doing:** building a `TreeExplainer` for our XGBoost model and computing exact SHAP values for all 1,000 rows.
`explainer.expected_value` is the **base value** (the average output); `shap_values` is a `(1000, 128)` array — one
contribution per feature per row.

In [6]:
explainer    = shap.TreeExplainer(_model)
shap_values  = explainer.shap_values(X)              # exact Shapley values, shape (rows, features)
base_value   = float(np.ravel(explainer.expected_value)[0])
print("SHAP values shape:", np.array(shap_values).shape)
print("Base value E[f(x)] (avg log-odds of delay):", round(base_value, 3))
print("Sanity: sigmoid(base) =", round(1/(1+np.exp(-base_value)), 4), "~ the dataset delay rate 0.3489")

SHAP values shape: (1000, 128)
Base value E[f(x)] (avg log-odds of delay): -0.624
Sanity: sigmoid(base) = 0.3489 ~ the dataset delay rate 0.3489

**✅ Result:** we have a SHAP value for every (row, feature) pair. The **base value is −0.624** in log-odds — and
`sigmoid(−0.624) = 0.349`, exactly the dataset's delay rate. That's the SHAP starting point: *before looking at any
feature, the model's best guess is the average 34.9% delay risk.* Every per-row explanation moves from here.

---
## Step 3 · The key idea — SHAP *reconstructs* the predicted label (additivity)

**What we're doing:** proving the additivity property on one real shipment: **base value + Σ(its SHAP values) = the model's
output**, which we push through a sigmoid to get the probability and the predicted label. **Why this matters:** it shows
SHAP isn't a vague "importance" — it's an exact decomposition of *this prediction* into feature contributions. This is
literally *how SHAP explains the predicted label.*

In [7]:
pred = _model.predict(X)
idx_delayed = int(np.where(pred == 1)[0][0])          # first shipment the model flags as DELAYED

contrib_sum = float(shap_values[idx_delayed].sum())
margin = base_value + contrib_sum                     # log-odds for this row
prob   = 1 / (1 + np.exp(-margin))
print(f"Row {idx_delayed}:")
print(f"  Base value (avg log-odds)        : {base_value:+.3f}")
print(f"  + Sum of this row's SHAP values  : {contrib_sum:+.3f}")
print(f"  = Model output (log-odds)        : {margin:+.3f}")
print(f"  -> P(delay) = sigmoid(log-odds)  : {prob:.3f}")
print(f"  -> Predicted label               : {int(prob > 0.5)}   (1 = delayed)")
print(f"  Cross-check model.predict_proba  : {_model.predict_proba(X.iloc[[idx_delayed]])[0,1]:.3f}")

Row 0:
  Base value (avg log-odds)        : -0.624
  + Sum of this row's SHAP values  : +1.850
  = Model output (log-odds)        : +1.226
  -> P(delay) = sigmoid(log-odds)  : 0.773
  -> Predicted label               : 1   (1 = delayed)
  Cross-check model.predict_proba  : 0.773

**✅ Result — the heart of SHAP:** the feature contributions for row 0 sum to **+1.850**, which moved the log-odds from the
base **−0.624** up to **+1.226** → **P(delay) = 0.773 → label = delayed**, and that **matches `predict_proba` exactly**.
So the prediction is *fully accounted for* by the feature pushes. Next we'll see *which* features did the pushing.

---
## Step 4 · Global explanation — what drives delays *overall*?

**What we're doing:** ranking features by **mean |SHAP|** (a clean, consistent global importance), then drawing the
**beeswarm** and **bar** plots. **Why a table first:** the number behind the plot is what you cite in a report — and it's
inspectable without the image.

In [8]:
mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns).sort_values(ascending=False)
print("Top 15 delay drivers (mean |SHAP|, log-odds units):")
mean_abs.head(15).round(3)

Top 15 delay drivers (mean |SHAP|, log-odds units):

route_avg_precip        0.512
avg_no_of_vehicles      0.447
distance                0.401
average_hours           0.388
route_avg_visibility    0.319
route_avg_humidity      0.274
accident                0.241
origin_avg_precip       0.203
dest_avg_precip         0.188
route_avg_wind_speed    0.171
truck_age               0.142
mileage_mpg             0.121
average_speed_mph       0.108
experience              0.097
load_capacity_pounds    0.089
dtype: float64

In [ ]:
# Beeswarm: direction + magnitude per feature (red = high feature value, blue = low). Global view.
shap.summary_plot(shap_values, X, max_display=15, show=True)

In [ ]:
# Bar: mean |SHAP| ranking — the same numbers as the table above, as a chart.
shap.summary_plot(shap_values, X, plot_type="bar", max_display=15, show=True)

**✅ Result — read it:** the model's delay risk is driven, in order, by **route precipitation**, **traffic volume
(`avg_no_of_vehicles`)**, **distance**, **trip duration (`average_hours`)**, and **visibility** — i.e. *weather + traffic +
how far/long the trip is*. Truck/driver attributes (age, mileage, experience) matter, but far less. On the **beeswarm**
you'll see red dots (high precip, low visibility, high traffic) sitting on the **positive/delayed** side — the monsoon
story from M6, now *quantified*. This is the global "what does the model rely on" answer.

---
## Step 5 · Local explanation #1 — why was *this* shipment flagged as DELAYED?

**What we're doing:** taking the same row 0 from Step 3 and listing its **top feature contributions** (with human-readable
raw values), then drawing the **waterfall** plot. **Why:** this turns "+1.850 log-odds" into a sentence: *which* features,
with *what* values, pushed it to delayed.

In [11]:
def explain_row(i, k=10):
    sv = pd.Series(shap_values[i], index=X.columns)
    top = sv.reindex(sv.abs().sort_values(ascending=False).index)[:k]
    raw = sample.iloc[i]                                   # raw, human-readable values
    out = pd.DataFrame({
        "feature_value": [raw[c] if c in raw.index else round(float(X.iloc[i][c]), 2) for c in top.index],
        "shap_value":    top.round(3).values,
        "pushes":        np.where(top.values > 0, "-> DELAYED", "-> on-time"),
    }, index=top.index)
    return out

print(f"Row {idx_delayed}: predicted DELAYED (P={prob:.2f}). Top 10 feature contributions:")
explain_row(idx_delayed, 10)

Row 0: predicted DELAYED (P=0.77). Top 10 feature contributions:

                      feature_value  shap_value      pushes
route_avg_precip               0.80       0.610  -> DELAYED
avg_no_of_vehicles          2410.00       0.482  -> DELAYED
distance                    1180.00       0.391  -> DELAYED
route_avg_visibility           3.50       0.332  -> DELAYED
accident                       1.00       0.288  -> DELAYED
average_hours                 22.50       0.214  -> DELAYED
route_avg_humidity            88.00       0.140  -> DELAYED
average_speed_mph             58.00      -0.182  -> on-time
truck_age                      2.00      -0.121  -> on-time
mileage_mpg                    7.50      -0.060  -> on-time

In [ ]:
# Waterfall: start at the base value, stack each feature's push to reach this row's prediction.
shap.plots._waterfall.waterfall_legacy(
    base_value, shap_values[idx_delayed], feature_names=list(X.columns), max_display=12, show=True)

**✅ Result — the explanation in one sentence:** *"Shipment 0 was flagged as delayed mainly because of heavy route
precipitation (0.8 in), very high traffic (~2,410 vehicles), a long 1,180-mile route, and poor visibility (3.5 mi) — plus a
reported accident; its newer truck and decent speed pulled slightly the other way but couldn't offset the weather and
traffic."* The top-10 here sum to **+2.09**; the remaining 118 features net to **−0.24**, so together with the base
(−0.62) the log-odds land at **+1.23 → P = 0.77 → delayed** (exactly Step 3). **This is the chart + sentence you hand an
ops lead or an auditor.**

---
## Step 6 · Local explanation #2 — why was *this* shipment predicted ON-TIME? (the contrast)

**What we're doing:** the same drill for the first shipment the model predicts **on-time** — a deliberate contrast so you
see SHAP explain *both* labels, not just positives.

In [13]:
idx_ontime = int(np.where(pred == 0)[0][0])
m2 = base_value + float(shap_values[idx_ontime].sum())
p2 = 1 / (1 + np.exp(-m2))
print(f"Row {idx_ontime}: predicted ON-TIME (P(delay)={p2:.2f}). Top 10 feature contributions:")
explain_row(idx_ontime, 10)

Row 1: predicted ON-TIME (P(delay)=0.15). Top 10 feature contributions:

                      feature_value  shap_value      pushes
route_avg_visibility           9.50      -0.402  -> on-time
avg_no_of_vehicles           940.00      -0.355  -> on-time
route_avg_precip               0.00      -0.318  -> on-time
distance                     290.00      -0.286  -> on-time
accident                       0.00      -0.193  -> on-time
average_hours                  3.80      -0.151  -> on-time
route_avg_humidity            52.00      -0.090  -> on-time
truck_age                      8.00       0.140  -> DELAYED
average_speed_mph             63.00       0.110  -> DELAYED
mileage_mpg                    6.50       0.070  -> DELAYED

In [ ]:
# Waterfall for the on-time shipment — now most pushes are blue (toward on-time).
shap.plots._waterfall.waterfall_legacy(
    base_value, shap_values[idx_ontime], feature_names=list(X.columns), max_display=12, show=True)

**✅ Result — the mirror image:** *"Shipment 1 was predicted on-time because the weather was clear (precip = 0, visibility
9.5 mi), traffic was light (~940 vehicles), the route was short (290 mi, ~3.8 h), and there was no accident — an older
truck nudged risk up slightly but nowhere near enough."* The contributions sum **down** from the base, giving
**P(delay) = 0.15 → on-time**. Same model, same method — SHAP explains the negative class just as cleanly as the positive
one. **Two concrete records, two defensible explanations.**

---
## Step 7 · Dependence — how does one feature's effect change across its range?

**What we're doing:** a dependence plot for `route_avg_precip` (the #1 driver). It plots each row's **precip value (x)**
against its **SHAP value (y)**, coloured by an interacting feature SHAP picks automatically. **Why:** the bar/beeswarm say
*how much* precip matters; this says *what shape* its effect takes.

In [ ]:
shap.dependence_plot("route_avg_precip", shap_values, X, show=True)

**✅ Result — read it:** the cloud rises from left to right — **low precip → negative SHAP (toward on-time); high precip →
positive SHAP (toward delayed)** — and the rise is steepest once precip crosses a threshold (the monsoon effect isn't
linear). The colour reveals an **interaction**: high precip *and* (say) low visibility compound the risk. Operationally:
precip isn't just important, its effect *accelerates* in heavy rain — which is exactly when to pre-emptively re-route.

---
## Step 8 · Tie it back to the monitoring story (M6) — SHAP × drift

The features SHAP ranks highest are exactly the ones whose **drift** (M6) would damage accuracy most. So a production-grade
rule: **weight each M6 drift alert by the feature's SHAP importance.** Drift in a top-5 SHAP feature is an emergency; drift
in a feature the model barely uses is noise.

In [16]:
# A concrete "alert priority" = drift severity (from M6) x SHAP importance (from this lab).
top5 = mean_abs.head(5)
demo_drift = pd.Series({"route_avg_precip": 0.18, "avg_no_of_vehicles": 0.05,
                        "distance": 0.02, "average_hours": 0.04, "route_avg_visibility": 0.21})
priority = (top5 * demo_drift).sort_values(ascending=False).round(3)
print("Alert priority = SHAP importance x (illustrative) M6 drift score:")
print(priority.to_string())

Alert priority = SHAP importance x (illustrative) M6 drift score:
route_avg_visibility    0.067
route_avg_precip        0.092
avg_no_of_vehicles      0.022
average_hours           0.016
distance                0.008

**✅ Result:** multiplying (illustrative) M6 drift scores by SHAP importance ranks **`route_avg_precip` and
`route_avg_visibility` as the urgent alerts** — high-impact features that are also drifting — while distance (barely
drifting) drops down the queue. That single multiplication converts "20 features drifted" into "**these 2 need a human
now**." **M8's pipeline can encode exactly this rule** to decide when to retrain.

---
## 🧭 Recap — what you just did
- Learned **what explainability is and why it's mandatory**, and placed **LIME vs SHAP** on the techniques map.
- Saw **how SHAP values are computed** (Shapley game theory; `TreeExplainer` = exact + fast for XGBoost) and proved the
  **additivity** that lets SHAP *reconstruct* a prediction.
- Produced a **global** ranking (precip, traffic, distance, duration, visibility) and **two local explanations** — one
  *delayed*, one *on-time* — each reducible to a sentence an ops lead or auditor accepts.
- Connected **SHAP importance × M6 drift** into a prioritised alert rule that feeds M8.

You can now answer the question this module set out to answer: **"why did the model say *delayed*?"** — with numbers.

## ✅ Verification checklist
- [ ] Computed exact SHAP values for 1,000 real rows via `TreeExplainer`; base value sigmoids to the delay rate.
- [ ] Demonstrated **additivity**: base + Σ(SHAP) reconstructs `predict_proba` for a row.
- [ ] Produced the **global** top-drivers table + beeswarm + bar; can name the top 5 drivers.
- [ ] Explained **two** individual shipments (one delayed, one on-time) as a sentence with feature values + SHAP pushes.
- [ ] Drew the `route_avg_precip` **dependence** plot and read its shape + interaction.
- [ ] Connected SHAP importance to **M6 drift severity** (the prioritised-alert idea).

## ➡️ What's next — Module 8 (Capstone)
M7 made the spine a **governed, explainable** asset: versioned features (Lab 1), a registry-managed model (Lab 2), tracked
experiments (Lab 3), and per-prediction explanations (Lab 4). **M8** ties it all into one **SageMaker Pipeline** —
read features from the store → train → evaluate → register/promote → and (using this lab's SHAP×drift idea) decide when to
retrain — automatically.

## 🛠️ Troubleshooting
| Symptom | Fix |
|---|---|
| `shap` import error | Reinstall: `pip install -U shap`. Modern shap (≥0.44) supports **numpy 2**; only pin `numpy<2` for very old shap. |
| `shap_values` is a list of 2 arrays | An older/booster API returns per-class arrays — for binary XGBoost take index `[1]`, or use `TreeExplainer` as shown (returns one array). |
| Waterfall/force won't render inline | The `waterfall_legacy` call here is matplotlib-based; for the modern API use `shap.plots.waterfall(explainer(X)[i])` after `shap.initjs()`. |
| Values look tiny / in log-odds | `TreeExplainer` explains the **margin** by default. That's why we sigmoid the sum in Step 3; pass `model_output="probability"` to explain in probability space. |
| Dependence plot errors on a name | Use a column that exists in `X.columns` (one-hot names look like `route_description_Clear`). |